# sum-back-expand-broadcast — worked example 2: Sum Backward — keepdim=False, unsqueeze then expand

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-back-expand-broadcast`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you sum along an axis with `keepdim=False`, the summed dimension is completely removed from the output. To back-propagate, you must first re-insert a size-1 axis at the original position using `unsqueeze(dim)`, then broadcast back to the input shape with `.expand(x.shape)`. This two-step pattern restores the missing dimension before broadcasting.

## Worked solution

**Step 1 — Trace the shapes.**
Suppose `x` has shape `(2, 5, 3)` and we call `x.sum(dim=1)`. The output has shape `(2, 3)` — axis 1 is gone.

**Step 2 — Re-insert the lost axis.**
`grad_out` has shape `(2, 3)`, but `x` expects a gradient of shape `(2, 5, 3)`. First we call `grad_out.unsqueeze(1)` to get shape `(2, 1, 3)`.

**Step 3 — Broadcast to full input shape.**
Now `.expand(x.shape)` stretches the size-1 middle dimension from 1 to 5, giving shape `(2, 5, 3)`. Every slice `x[:, j, :]` gets the same gradient value.

**Step 4 — Why this is correct.**
Each `x[b, j, c]` contributed to exactly one output `out[b, c]` with local derivative 1, so `grad_in[b, j, c] = grad_out[b, c]` for all j — which is exactly what expand achieves.

In [ ]:
import torch as t

def sum_back_no_keepdim(grad_out: t.Tensor, x: t.Tensor, dim: int) -> t.Tensor:
    """Backward for out = x.sum(dim=dim, keepdim=False)."""
    # Step 1: restore the removed axis as size-1
    g = grad_out.unsqueeze(dim)
    # Step 2: broadcast to original input shape
    return g.expand(x.shape)

# Demonstrate
t.manual_seed(0)
x = t.randn(2, 5, 3)
out = x.sum(dim=1)                         # shape (2, 3)
grad_out = t.ones_like(out)                # shape (2, 3)
grad_in = sum_back_no_keepdim(grad_out, x, dim=1)
print('x.shape:', x.shape)                # (2, 5, 3)
print('grad_out.shape:', grad_out.shape)  # (2, 3)
print('grad_in.shape:', grad_in.shape)    # (2, 5, 3)
print('First row grad_in[0]:')           # all same along dim=1
print(grad_in[0])